# Session 2 — Useful Python: automating the boring stuff

We're going to open the folder `sample_data/`, which mimics a real
researcher's downloads folder — WTO document dumps with tangled
filenames, tariff schedules, meeting notes, a couple of duplicates. We
build up, over the session, a small script that sorts all of it into
tidy subfolders and writes a short summary of what was done.

**How the session is shaped**

| Block            | What happens                                       | Time   |
|------------------|----------------------------------------------------|--------|
| Live coding 4    | `pathlib`, exploring and filtering a folder        | 20 min |
| Exercise 4       | you write code                                     | 20 min |
| Live coding 5    | making folders, copying, moving, writing files     | 20 min |
| Capstone         | organise the whole folder + summary (in-class start, homework finish) | 20+ min |

The **capstone** requires nothing from the Homework tier of any block —
only Core and Stretch content, from both sessions.


## Making sure `sample_data/` is there

The folder is generated by `setup_sample_data.py` sitting next to this
notebook. If the next cell complains, open a terminal in VS Code
(**Terminal → New Terminal**) and run:

```
python setup_sample_data.py
```

Then run the next cell again.


In [ ]:
from pathlib import Path

data_dir = Path("sample_data")

if not data_dir.exists():
    raise FileNotFoundError(
        "sample_data/ not found — run 'python setup_sample_data.py' in the terminal, then re-run this cell."
    )

files = list(data_dir.iterdir())
print(f"Found {len(files)} files in {data_dir.resolve()}")


Have a look at the first few filenames, just to get a feel for the
mess:


In [ ]:
for f in files[:8]:
    print(f.name)


---

## Part 4 — Exploring a folder with `pathlib`

*(Live coding — roughly 20 minutes.)*

`pathlib` is Python's modern way of talking about files and folders.
The core idea: a `Path` is a small object that knows a bunch of useful
things about a file's location. You never manipulate paths as raw
strings once you know `Path`.


In [ ]:
from pathlib import Path

p = Path("sample_data") / "WT-DS316-AB-R.txt"
print(p)


The `/` operator between paths joins them safely — it inserts the right
separator (`\` on Windows, `/` on Mac and Linux). You never have to
worry about that yourself.

Every `Path` knows several useful things about itself:


In [ ]:
p = Path("sample_data") / "WT-DS316-AB-R.txt"

print("full path :", p)
print("name      :", p.name)     # WT-DS316-AB-R.txt
print("stem      :", p.stem)     # WT-DS316-AB-R  (name without suffix)
print("suffix    :", p.suffix)   # .txt
print("parent    :", p.parent)   # sample_data
print("exists    :", p.exists())


### Listing what's inside a folder

`iterdir()` gives you every entry in a folder, one `Path` at a time.


In [ ]:
data_dir = Path("sample_data")

for f in data_dir.iterdir():
    print(f.name)


Combined with `for` and `if`, this is already enough to answer real
questions — like *"how many CSVs are in here?"*.


In [ ]:
count_csv = 0

for f in data_dir.iterdir():
    if f.suffix == ".csv":
        count_csv = count_csv + 1

print(f"{count_csv} CSV files")


### Filtering with string methods

`Path.name` is a string, so all the string methods from Session 1 work
on it — `.startswith`, `.endswith`, `.lower()`, `in`, and so on.


In [ ]:
for f in data_dir.iterdir():
    if "asbestos" in f.name.lower():
        print(f.name)


In [ ]:
official_looking = []

for f in data_dir.iterdir():
    name = f.name
    if name.startswith("WT-") or name.startswith("G_") or name.startswith("TN_"):
        official_looking.append(name)

print(len(official_looking), "files look officially named")
for n in official_looking[:5]:
    print("  ", n)


### Counting by category, into a dictionary

This is a classic pattern: loop, decide, count in a dictionary.


In [ ]:
counts = {}

for f in data_dir.iterdir():
    suffix = f.suffix
    if suffix in counts:
        counts[suffix] = counts[suffix] + 1
    else:
        counts[suffix] = 1

print(counts)


Same pattern, more compact, using `.get(key, default)` — a dictionary
method that returns `default` when the key isn't there yet.


In [ ]:
counts = {}

for f in data_dir.iterdir():
    counts[f.suffix] = counts.get(f.suffix, 0) + 1

print(counts)


### `glob` — matching filenames with a pattern

`glob("*.csv")` returns everything in the folder whose name matches
that pattern. `*` means "anything". It's often quicker than an
`iterdir + if`.


In [ ]:
csvs = list(data_dir.glob("*.csv"))
print(len(csvs), "CSV files")

for c in csvs:
    print(" ", c.name)


---

## Exercise Block 4 — Exploring the folder

*(About 20 minutes.)*

Assume the setup cell at the top has been run so that `data_dir` and
`files` are defined. If not, run it now.

### Core (everyone)

1. Print the total number of files in `sample_data/` using
   `iterdir()` and a counter. (Don't use `len(files)` — practice the
   loop.)
2. Print the `.stem` of each file whose `.suffix` is `.txt`.
3. Count how many files contain the string `"DS"` anywhere in their
   name (case-insensitive). Print the count.
4. Using `.glob(...)`, list all files ending in `.csv` and print their
   `.name` values, one per line.

### Stretch (if you have time)

1. Build a dictionary `counts_by_suffix` mapping each suffix (like
   `".txt"` or `".csv"`) to how many files carry it. Print it.
2. Build a list `probably_renamed` of filenames that do **not**
   start with `WT-`, `G_`, or `TN_` — using `startswith` and `or`.
   Print the length, and the first three items.
3. Print the **longest filename** in the folder (measured by
   `len(f.name)`) and its length. Only Session-1 material is needed.

### Homework (harder, optional — not needed for the capstone)

Write a function `filter_by(files, contains=None, suffix=None)` that
takes a list of `Path` objects and returns only those whose name
contains the given substring (if provided) *and* whose suffix matches
(if provided). Both arguments should default to `None`, meaning "don't
filter on this criterion". Test it with three combinations: only
`suffix=".txt"`, only `contains="ds"`, and both together.


In [ ]:
from pathlib import Path

data_dir = Path("sample_data")
files = list(data_dir.iterdir())

# Core
# your code here


# Stretch
# your code here


# Homework (optional)
# your code here


---

## Part 5 — Making folders, copying, moving, writing

*(Live coding — roughly 20 minutes.)*

We now leave read-only territory. Everything from here on changes
things on disk. That's fine — we only touch new folders we create
ourselves, and the source `sample_data/` stays untouched.


### Making a folder

`mkdir()` creates a folder. Two arguments come up all the time:

- `parents=True` means "create any missing parent folders too".
- `exist_ok=True` means "don't complain if the folder already exists".

Together, they let you write "make sure this folder exists" in one
line, safely.


In [ ]:
target = Path("organised") / "appellate_body"
target.mkdir(parents=True, exist_ok=True)

print(target.exists())


### Copying a file

`shutil.copy(source, destination)` copies a file. Both arguments can be
`Path` objects. The destination folder has to exist first — that's
what the `mkdir` above is for.


In [ ]:
import shutil

source = Path("sample_data") / "WT-DS316-AB-R.txt"
destination = Path("organised") / "appellate_body" / source.name

shutil.copy(source, destination)
print(destination.exists())


### Moving and renaming

`shutil.move(source, destination)` moves a file. `Path.rename(new)`
renames it in place. Both do essentially the same thing when the target
is in the same folder — pick whichever reads more naturally.


In [ ]:
old = destination
new = destination.with_name("renamed_example.txt")

old.rename(new)

print("old exists?", old.exists())
print("new exists?", new.exists())


### Writing a text file

The simplest form: `Path.write_text(...)` writes a string to a file,
overwriting any existing content.


In [ ]:
report = Path("organised") / "summary.txt"

report.write_text(
    "Organisation summary\n"
    "--------------------\n"
    "Files handled so far: 1\n"
)

print("wrote", report)


### Reading a text file back

`Path.read_text()` reads the whole file into a single string.


In [ ]:
content = report.read_text()
print(content)


### Building a mini organiser

Now the pieces come together. We loop through every file in
`sample_data/`, decide which category it belongs to, make sure the
matching folder exists, and copy the file in. Nothing new here — just
Session 1 patterns applied to disk.


In [ ]:
def classify(filename):
    name = filename.lower()
    if "ab" in name and "ds" in name:
        return "appellate_body"
    elif "panel" in name:
        return "panel_report"
    elif name.endswith(".csv") or "tariff" in name:
        return "tariff_schedule"
    elif "min" in name or name.startswith("tn_"):
        return "ministerial"
    elif "agreement" in name or "understanding" in name or "gatt" in name:
        return "legal_text"
    else:
        return "unclassified"


out_root = Path("organised_demo")

for f in data_dir.iterdir():
    category = classify(f.name)
    dest_folder = out_root / category
    dest_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, dest_folder / f.name)

print("done. contents of organised_demo/:")
for sub in out_root.iterdir():
    n = len(list(sub.iterdir()))
    print(f"  {sub.name}: {n} files")


This little script is the whole point of the session. Everything else
in Part 5 is just filling in the details for what you'd want to add
for real use — better classification, a proper summary file, some
handling of duplicates.


---

## Exercise Block 5 — Capstone

*(Start in class, finish as homework. Core is small so it fits in about
20 minutes. Stretch and Homework are for after class.)*

You'll be modifying folders on disk. Two safety notes:

- Only write into new folders (like `organised/` or `my_test/`), never
  into `sample_data/`. Leave the source folder alone.
- If something goes wrong, re-running the setup script recreates
  `sample_data/` from scratch.

### Core (everyone) — build your own tiny organiser

1. Create a folder called `my_organised/` next to this notebook.
2. Inside it, create two subfolders: `csv_files/` and `txt_files/`.
3. Loop through `sample_data/`. For every `.csv` file, copy it into
   `my_organised/csv_files/`. For every `.txt` file, copy it into
   `my_organised/txt_files/`. Anything else, skip.
4. Print how many files ended up in each subfolder, using `.iterdir()`
   and a counter.

### Stretch — a real summary file

1. Extend your organiser: instead of only `.csv` and `.txt`, sort by
   the categories from the live-coding example (`appellate_body`,
   `panel_report`, `tariff_schedule`, `ministerial`, `legal_text`,
   `unclassified`). Reuse or adapt the `classify` function from above.
2. As you copy, keep a **running dictionary** `counts` that maps each
   category to how many files ended up in it. Do this by updating the
   dictionary inside the loop, not by counting folder contents at the
   end.
3. After the loop, write a summary file `my_organised/summary.txt`
   with one line per category (`<category>: <count>`), and a final
   line `total: <n>`. Read it back with `read_text()` and print it.

### Homework (harder, optional) — handle collisions and duplicates

1. `sample_data/` contains two files that look like the same shrimp-
   turtle report under different names (`ds58_shrimp_turtle_AB.txt` and
   `WT-DS58-AB-R_shrimp_turtle.txt`). Your organiser currently copies
   both. Extend it so that when copying, if the target folder already
   contains a file whose `stem` starts with the same case number (e.g.
   both start with something derivable from `"ds58"` / `"ds316"`), the
   new file is copied under a modified name — append `_dup` to the
   stem — instead of overwriting.
2. Add a new section to the summary listing every case where a
   `_dup` copy was made.
3. Package the whole thing into a single function
   `organise(source_folder, target_folder)` that does all of the above
   and returns the counts dictionary. Test it by calling it a second
   time on the already-organised output — it should be a no-op modulo
   the `_dup` handling.

Good luck. When you're stuck, print things. Half of Python is finding
out what `print(some_variable)` is really showing you, not what you
imagined it would show.


In [ ]:
from pathlib import Path
import shutil

data_dir = Path("sample_data")

# Core
# your code here


# Stretch
# your code here


# Homework (optional)
# your code here


---

## Wrap-up

Six hours ago you'd never opened a notebook. If you finished the Core
tier of the capstone, you've written a small script that automates a
task that would take an afternoon by hand — for one hundred files, or
one thousand, or ten thousand, it takes exactly the same amount of
typing.

The pattern you now know — **loop over a folder, decide something
about each item, act on it, write down what happened** — is the shape
of a huge fraction of everyday research automation. Nothing about it
was specific to trade law; swap in a folder of interview transcripts,
of PDFs from a court archive, of scraped news articles, and the
scaffolding is the same.
